# Curva de TIRs — Obligaciones Negociables (USD)

Dot plot de **TIR vs. fecha de vencimiento** para todas las ONs en USD que listan en PPI con datos completos.

**Cómo lo arma:**
1. Auth con cache de sesión: la primera vez pide OTP (que llega por mail); las siguientes corridas reusan el JWT mientras esté vivo.
2. Universo de ONs: `Item/Search?q=ON` filtrando `tipoItem.id == 140` (no se usa `get_tickers_list` porque tu cuenta lo tiene restringido).
3. Para cada ON USD operable: trae flujos teóricos y último precio histórico.
4. **TIR se calcula localmente** (PPI no la expone como campo): bisección sobre la suma de flujos descontados.
5. Plot con Plotly.

In [1]:
import os
from datetime import datetime, timezone, date, timedelta
from pathlib import Path

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from py_ppi_arg import PPI

load_dotenv(Path("..") / ".env")

True

In [2]:
# Si el cache local tiene un JWT vivo, no pide OTP. Si no, te lo va a pedir por input().
# El código llega por mail al disparar el login.
app = PPI(
    user=os.environ["PPI_USER"],
    password=os.environ["PPI_PASSWORD"],
    remember_device=True,
)
print("Auth OK. Token expira:", app._token_expires_at)

Auth OK. Token expira: 2026-05-11T11:11:24-03:00


## 1. Universo de ONs en USD

In [3]:
from collections import Counter

all_items = app.search_tickers(short_ticker="ON")["payload"]
ons = [
    i for i in all_items
    if isinstance(i.get("tipoItem"), dict)
    and i["tipoItem"].get("id") == 140
    and i.get("operable24")
]
print(f"ONs operables 24h: {len(ons)}")


def quote_market(item):
    """Identifica el mercado de cotización a partir del campo `moneda` del item."""
    m = item.get("moneda") or {}
    desc = (m.get("descripcion") or "").lower()
    if "mep" in desc:
        return "USD MEP"
    if "ccl" in desc or m.get("id") == 10001:
        return "USD CCL"
    if "peso" in desc or m.get("id") == 10000:
        return "ARS"
    return "OTRO"


print("Distribución por moneda de cotización:", Counter(quote_market(i) for i in ons))

ONs operables 24h: 1277
Distribución por moneda de cotización: Counter({'USD MEP': 439, 'ARS': 433, 'USD CCL': 405})


## 2. Flujos + último precio para cada ON

~800 calls (2 por ON). Tarda unos minutos. Las ONs sin flujos o sin cotización reciente se descartan automáticamente.

In [4]:
today = date.today()
PRICE_LOOKBACK_DAYS = 30   # ampliar para capturar ONs menos líquidas
date_from = (today - timedelta(days=PRICE_LOOKBACK_DAYS)).strftime("%Y-%m-%d")
date_to = today.strftime("%Y-%m-%d")

# Limit para iteración rápida en debug. None = correr todas.
LIMIT = None

rows = []
errors = 0
iterable = ons[:LIMIT] if LIMIT else ons
for n, item in enumerate(iterable, 1):
    sid = item["id"]
    ticker = item.get("ticker")
    descripcion = item.get("descripcion")
    try:
        td = app.get_technical_data_bonds(settlement=app.settlements.T2, item_id=str(sid)).get("payload") or {}
        flujos = td.get("flujosDeFondosTeoricos") or []
        if not flujos:
            continue
        hist = app.get_historic_data(
            item_id=str(sid),
            settlement=app.settlements.T2,
            date_from=date_from,
            date_to=date_to,
        ).get("payload") or []
        if not hist:
            continue
        last = hist[-1]
        price = last.get("ultOperado") or last.get("cierreAnterior")
        if not price or price <= 0:
            continue
        rows.append({
            "id": sid,
            "ticker": ticker,
            "descripcion": descripcion,
            "fechaVencimiento": td.get("fechaVencimiento"),
            "price": price,
            "price_date": last.get("fechaCotizacion"),
            "volumen": last.get("volumen") or 0,
            "moneda_cotizacion": quote_market(item),
            "moneda_emision": "USD" if "U$S" in (descripcion or "") else (flujos[0].get("moneda") or "?").upper(),
            "flujos": flujos,
            "intereses": td.get("intereses"),
        })
    except Exception:
        errors += 1
    if n % 50 == 0:
        print(f"  {n}/{len(iterable)}  con datos: {len(rows)}  errores: {errors}")

print(f"\nFinal: {len(rows)} ONs con flujos + precio (de {len(iterable)} probadas, {errors} errores)")

  300/1277  con datos: 110  errores: 0
  350/1277  con datos: 120  errores: 0
  400/1277  con datos: 141  errores: 0
  500/1277  con datos: 194  errores: 0
  550/1277  con datos: 225  errores: 0
  700/1277  con datos: 285  errores: 0
  800/1277  con datos: 327  errores: 0
  900/1277  con datos: 379  errores: 0
  950/1277  con datos: 393  errores: 0
  1050/1277  con datos: 434  errores: 0
  1100/1277  con datos: 470  errores: 0
  1150/1277  con datos: 502  errores: 0
  1200/1277  con datos: 538  errores: 0
  1250/1277  con datos: 572  errores: 0

Final: 586 ONs con flujos + precio (de 1277 probadas, 0 errores)


## 3. Calcular TIR

Resolvemos por bisección: encontrar `r` tal que `Σ flow.total / (1+r)^((flow.fechaCorte - hoy)/365) == price`. Solo se cuentan flujos futuros.

In [8]:
def npv(rate, flows, today_dt):
    s = 0.0
    for f in flows:
        fd = pd.to_datetime(f["fechaCorte"]).to_pydatetime()
        if fd.tzinfo is not None:
            fd = fd.astimezone(timezone.utc).replace(tzinfo=None)
        days = (fd - today_dt).days
        if days <= 0:
            continue
        t = days / 365.0
        s += f["total"] / (1 + rate) ** t
    return s

def ytm(flows, price, today_dt, lo=-0.5, hi=5.0, tol=1e-6, max_iter=200):
    """YTM por bisección. Devuelve None si no encuentra raíz en [lo, hi]."""
    f_lo = npv(lo, flows, today_dt) - price
    f_hi = npv(hi, flows, today_dt) - price
    if f_lo * f_hi > 0:
        return None
    for _ in range(max_iter):
        mid = (lo + hi) / 2
        f_mid = npv(mid, flows, today_dt) - price
        if abs(f_mid) < tol:
            return mid
        if f_lo * f_mid < 0:
            hi = mid
            f_hi = f_mid
        else:
            lo = mid
            f_lo = f_mid
    return (lo + hi) / 2

today_dt = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0, tzinfo=None)

for r in rows:
    r["tir"] = ytm(r["flujos"], r["price"], today_dt)

df = pd.DataFrame([r for r in rows if r["tir"] is not None])
df["fechaVencimiento"] = pd.to_datetime(df["fechaVencimiento"], errors="coerce", utc=True).dt.tz_convert(None)
df["price_date"] = pd.to_datetime(df["price_date"], errors="coerce", utc=True).dt.tz_convert(None)
df["tir_pct"] = df["tir"] * 100
df = df.dropna(subset=["fechaVencimiento", "tir_pct"]).copy()
df = df[df["fechaVencimiento"] >= pd.Timestamp.now().normalize()]
df = df.sort_values("fechaVencimiento").reset_index(drop=True)
print(f"Filas finales: {len(df)}")
print("Por moneda de cotización:", df["moneda_cotizacion"].value_counts().to_dict())
print("Por moneda de emisión:   ", df["moneda_emision"].value_counts().to_dict())
df[["ticker", "moneda_cotizacion", "moneda_emision", "fechaVencimiento", "price", "tir_pct", "volumen"]].head(20)

Filas finales: 111
Por moneda de cotización: {'USD MEP': 49, 'USD CCL': 43, 'ARS': 19}
Por moneda de emisión:    {'USD': 110, '$': 1}


,ticker,moneda_cotizacion,moneda_emision,fechaVencimiento,price,tir_pct,volumen
0,VSCHO,ARS,USD,2026-06-06 20:29:10.320,135000.000000,178.693062,0.000000
1,MTCGD,USD MEP,USD,2026-06-30 16:01:38.330,104.900002,-14.103398,255101.187500
2,MTCGC,USD CCL,USD,2026-06-30 16:01:38.330,101.400002,10.042380,312.359985
3,LIC6D,USD MEP,USD,2026-07-02 20:21:26.660,33.419998,31.856433,0.000000
4,PZCAO,ARS,USD,2026-07-27 19:53:25.940,137200.000000,30.925454,0.000000
5,JNC5D,USD MEP,USD,2026-10-09 20:16:49.013,102.400002,1.454248,0.000000
6,GN37O,ARS,USD,2026-11-11 15:38:54.547,91162.320312,17.330657,0.000000
7,VSCIO,ARS,$,2026-12-06 18:17:37.200,137500.000000,10.016422,0.000000
8,PQCKO,ARS,USD,2026-12-07 16:08:33.220,68230.000000,86.315766,0.000000
9,SIC1C,USD CCL,USD,2026-12-09 19:25:53.160,97.500000,16.915156,0.000000


## 4. Filtros

Ajustá las dos listas según qué subset querés ver. Por defecto: cotiza en pesos + hard-dollar (paga USD).

In [9]:
# ====== FILTROS ======
# moneda_cotizacion: dónde se opera ("ARS", "USD MEP", "USD CCL", "OTRO")
# moneda_emision:    en qué paga el bono ("USD" = hard-dollar; "PESO"/"ARS" = pesos; etc.)
#
# Ejemplos:
#   QUOTE_CURRENCY = None  # None = todas las monedas de cotización                   -> solo las que cotizan en pesos
#   QUOTE_CURRENCY = ["USD MEP", "USD CCL"]    -> solo las dólar
#   QUOTE_CURRENCY = None                      -> sin filtro
#
#   PAYMENT_CURRENCY = ["USD"]                 -> solo hard-dollar (paga en USD)
#   PAYMENT_CURRENCY = None                    -> sin filtro

QUOTE_CURRENCY = None  # None = todas las monedas de cotización
PAYMENT_CURRENCY = ["USD"]
TIR_RANGE = (-20, 25)   # descartar outliers por flujos ARS incompletos

view = df.copy()
if QUOTE_CURRENCY:
    view = view[view["moneda_cotizacion"].isin(QUOTE_CURRENCY)]
if PAYMENT_CURRENCY:
    view = view[view["moneda_emision"].isin(PAYMENT_CURRENCY)]
view = view[view["tir_pct"].between(*TIR_RANGE)]
view = view.reset_index(drop=True)

print(f"{len(view)} ONs tras filtro: cotiza en {QUOTE_CURRENCY}, paga en {PAYMENT_CURRENCY}")
view[["ticker", "descripcion", "moneda_cotizacion", "moneda_emision", "fechaVencimiento", "price", "tir_pct", "volumen"]].head(15)

95 ONs tras filtro: cotiza en None, paga en ['USD']


,ticker,descripcion,moneda_cotizacion,moneda_emision,fechaVencimiento,price,tir_pct,volumen
0,MTCGD,ON Mastellone Hermanos S.A. 10.95% V30/06/26,USD MEP,USD,2026-06-30 16:01:38.330,104.900002,-14.103398,255101.187500
1,MTCGC,ON Mastellone Hermanos S.A. 10.95% V30/06/26,USD CCL,USD,2026-06-30 16:01:38.330,101.400002,10.042380,312.359985
2,JNC5D,ON INV. JURAMENTO CL.5 V09/10/26 U$S CG,USD MEP,USD,2026-10-09 20:16:49.013,102.400002,1.454248,0.000000
3,GN37O,ON GENNEIA CL. 37 U$S VTO.11/11/2026,ARS,USD,2026-11-11 15:38:54.547,91162.320312,17.330657,0.000000
4,SIC1C,ON SIDERSA CL1 V09/12/26,USD CCL,USD,2026-12-09 19:25:53.160,97.500000,16.915156,0.000000
5,SIC1D,ON SIDERSA CL1 V09/12/26,USD MEP,USD,2026-12-09 19:25:53.160,102.599998,6.821945,38989.281250
6,AERBC,ON AEROP ARG 2000 11 V15/12/26,USD CCL,USD,2026-12-15 15:14:04.743,101.349998,6.895914,0.000000
7,AERBD,ON AEROP ARG 2000 11 V15/12/26,USD MEP,USD,2026-12-15 15:14:04.743,103.800003,2.615656,16722.199219
8,CS44C,ON CRESUD S28 CL44 V17/01/27 U$S CG,USD CCL,USD,2027-01-17 19:34:40.773,99.860001,9.268142,0.000000
9,CS44D,ON CRESUD S28 CL44 V17/01/27 U$S CG,USD MEP,USD,2027-01-17 19:34:40.773,102.400002,5.265402,15466.059570


## 5. Dot plot

In [13]:
import plotly.graph_objects as go
import numpy as np

title_filter = (
    f"cotización: {', '.join(QUOTE_CURRENCY) if QUOTE_CURRENCY else 'todas'}"
    f"  ·  emisión: {', '.join(PAYMENT_CURRENCY) if PAYMENT_CURRENCY else 'todas'}"
)

COLORS = {"ARS": "#636EFA", "USD CCL": "#EF553B", "USD MEP": "#00CC96"}

view_plot = view.copy()
view_plot["moneda_cotizacion"] = view_plot["moneda_cotizacion"].astype(str)
view_plot["fechaVencimiento"]  = pd.to_datetime(view_plot["fechaVencimiento"])

# Escalar volumen a tamaño de marker: raíz cuadrada + mínimo de 6px
vol = view_plot["volumen"].fillna(0).clip(lower=0)
vol_max = vol.max() if vol.max() > 0 else 1
view_plot["_size"] = 6 + 30 * np.sqrt(vol / vol_max)

fig = go.Figure()
for cotiz, grp in view_plot.groupby("moneda_cotizacion"):
    fig.add_trace(go.Scatter(
        x=grp["fechaVencimiento"],
        y=grp["tir_pct"],
        mode="markers+text",
        name=cotiz,
        marker=dict(
            size=grp["_size"].tolist(),
            color=COLORS.get(cotiz),
            line=dict(width=1, color="DarkSlateGrey"),
        ),
        text=grp["ticker"],
        textposition="top center",
        customdata=grp[["ticker", "descripcion", "price", "volumen", "moneda_emision"]].values,
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "%{customdata[1]}<br>"
            "TIR: <b>%{y:.2f}%</b><br>"
            "Precio: %{customdata[2]:,.0f}<br>"
            "Volumen: %{customdata[3]:,.0f}<br>"
            "Emisión: %{customdata[4]}<extra></extra>"
        ),
    ))

fig.update_layout(
    title=f"Curva de TIRs — Obligaciones Negociables<br><sup>{title_filter}  ·  tamaño = volumen operado</sup>",
    xaxis_title="Fecha de vencimiento",
    yaxis_title="TIR (%)",
    yaxis_ticksuffix="%",
    height=600,
    template="plotly_white",
    legend_title="Cotización",
)
fig.show()


## 6. Persistir TIRs a disco

Guarda el DataFrame completo (sin filtrar por moneda) a `output/on_tir.parquet` para que otros notebooks (ej. análisis de riesgo cruzando con balances) puedan reutilizar los datos sin volver a llamar a la API.

In [ ]:
import json

out_path = Path("..") / "output" / "on_tir_dotplot.parquet"  # no sobreescribe el de fetch_ons.py
out_path.parent.mkdir(parents=True, exist_ok=True)

# Parquet no maneja listas anidadas heterogéneas — serializo flujos/intereses a JSON.
df_to_save = df.copy()
df_to_save["flujos"] = df_to_save["flujos"].apply(json.dumps)
df_to_save["intereses"] = df_to_save["intereses"].apply(json.dumps)
df_to_save.to_parquet(out_path, index=False)
print(f"Guardado: {out_path.resolve()}  ({len(df_to_save)} filas)")

Guardado: /Users/mbasualdo/Documents/py-PPI-analysis-main/output/on_tir_dotplot.parquet  (111 filas)
